# De Bruijn graphs

You will be implementing one of the primary assembly algorithms from short-read data that is used today. We will implement a simple form of the algorithm where we _assume perfect sequencing_. That is, everything is sequenced exactly once and there are no errors or variants in the sequencing. 

A graph is composed of **nodes** and **edges** and we will need to develop a data strcture to track edges between nodes in our graph. We have provided the basic class structure as well as descriptions of functions to `add_edge` and `remove_edge` from the graph. You will need to implement these functions in order to then build the de Bruijn graph. 

In our implementation below, we use a `defaultdict` data structure to hold a list of all edges in the graph where all "right" nodes connected to a "left" node are stored in a list for that node.

```
build_debruijn_graph:
define substring length k and input string
For each k-length substring of input:
  split k mer into left and right k-1 mer
  add k-1 mers as nodes with a directed edge from left k-1 mer to right k-1 mer
```

---
## Eulerian walk

To continue our implementation from last class, we will use our De Bruijn graph to output a valid sequence from the assembly. This is implemented as a recursive algorithm by considering all valid edges. You will notice that as you change $k$, we are able to better recapitulate our sequence depending on how repetitive it is. In a more complex implementation of a Eulerian walk there are heuristics and defined rules for determining the validity of traversing a specific edge in the graph to result in a full graph-traversal. One of these methods is to traverse the graph in a depth first manner to avoid sectioning off any part of the graph in the traversal. In our implementation we will ignore these for simplicity.

```
eulerian_walk:
Beginning at first_node as node

For node:
    follow a random valid edge from node
    remove edge
    recurse
```


In [2]:
def read_fastq(filename):
    """Read sequences from a FASTQ file.

    Args:
        filename (str): Path to FASTQ file.

    Returns:
        list: List of DNA sequence strings (quality scores ignored).

    Example:
        >>> reads = read_fastq("test_reads.fastq")
        >>> len(reads)
        10
    """
    sequences = []
    with open(filename, 'r') as f:
        line_count = 0
        for line in f:
            line_count += 1
            if line_count % 4 == 2:  # Sequence line in FASTQ format
                sequences.append(line.strip())
    return sequences

In [3]:
"""De Bruijn Graph Genome Assembly Module.

This module provides classes and functions for constructing De Bruijn graphs
from sequencing reads and performing genome assembly using Eulerian path
traversal.
"""

from collections import defaultdict
import random


class DeBruijnGraph:
    """Main class for De Bruijn graphs and genome assembly.

    This class builds De Bruijn graphs from sequencing reads and performs
    genome assembly by finding Eulerian paths through connected components
    of the graph.

    Attributes:
        graph (defaultdict): Adjacency list representation of graph edges.
            Keys are (k-1)-mers, values are lists of adjacent (k-1)-mers.
        k (int): The k-mer size used for graph construction.

    Example:
        >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
        >>> dbg = DeBruijnGraph(reads, k=6)
        >>> contigs = dbg.assemble_contigs(seed=42)
        >>> len(contigs) > 0
        True
    """

    def __init__(self, reads, k):
        """Initialize De Bruijn graph from sequencing reads.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer size for graph construction.

        Example:
            >>> reads = ["ATGGCG", "GCGTGC", "TGCAAC"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> len(dbg.graph) > 0
            True
        """
        self.graph = defaultdict(list)
        self.k = k

        self.prefix_counts = defaultdict(int)
        self.suffix_counts = defaultdict(int)

        self.build_graph_from_reads(reads, k)

    def add_edge(self, left, right):
        """Add a directed edge to the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> "TGG" in dbg.graph["ATG"]
            True
        """
        self.graph[left].append(right)

    def remove_edge(self, left, right):
        """Remove a directed edge from the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> dbg.remove_edge("ATG", "TGG")
            >>> len(dbg.graph["ATG"])
            0
        """
        self.graph[left].remove(right)

    def build_graph_from_reads(self, reads, k):
        """Build De Bruijn graph from multiple sequencing reads.

        Extracts all k-mers from all reads and adds edges between
        consecutive (k-1)-mers within each k-mer.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer length for graph construction.

        Example:
            >>> reads = ["ATGGC", "TGGCA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.build_graph_from_reads(reads, 4)
            >>> "ATG" in dbg.graph
            True
        """
        if not reads:
            return

        #for check if k is too big
        ignored = 0
        total = len(reads)

        for read in reads:
            if len(read) < k:
                ignored += 1
                continue


            for i in range(len(read) - k + 1):
                kmer = read[i : i + k]
                left = kmer[:-1]
                right = kmer[1:]

                #degree counting
                self.prefix_counts[left] += 1
                self.suffix_counts[right] += 1

                #add edge
                self.add_edge(left, right)

                #ensure suffix-only nodes exist as key in adjacency
                _ = self.graph[right]

        #print % ignored
        if total > 0:
            pct = (ignored / total) * 100
            print(f"Ignored {ignored}/{total} reads ({pct:.2f}%) because read_length < k")


    def eulerian_walk(self, node, graph, seed=None):
        """Perform recursive Eulerian walk on a graph component.

        This is a recursive function that follows all edges from a node
        to traverse the graph, building a path in reverse order.

        Args:
            node (str): Current node to traverse from.
            graph (defaultdict): Graph or subgraph to traverse.
            seed (int, optional): Seed for random edge selection.

        Returns:
            list: List of (k-1)-mers traversed (in reverse order).

        Example:
            >>> reads = ["ATGGCG"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> graph_copy = defaultdict(list, dbg.graph)
            >>> tour = dbg.eulerian_walk("ATG", graph_copy, seed=42)
            >>> len(tour) > 0
            True
        """
        if seed is not None:
            random.seed(seed)

        tour = []

        def _walk(current_node):
            #while there are outgoing edges
            while graph[current_node]:
                #choose next edge, random choice of seed set, else pop last
                if seed is not None:
                    next_node = random.sample(graph[current_node])
                    graph[current_node].remove(next_node)
                else:
                    next_node = random.sample(graph[current_node])
                _walk(next_node)
            tour.append(current_node)

        _walk(node)
        return tour


    def assemble_contigs(self, seed=None):
        """Assemble all contigs from the De Bruijn graph.

        Finds all connected components and generates an Eulerian path
        for each component, producing multiple assembled contigs.

        Args:
            seed (int, optional): Random seed for reproducible assembly.

        Returns:
            list: List of assembled contig sequences (DNA strings).

        Example:
            >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
            >>> dbg = DeBruijnGraph(reads, k=6)
            >>> contigs = dbg.assemble_contigs(seed=42)
            >>> all(isinstance(c, str) for c in contigs)
            True
        """
        #copy graph and ensure graph and graph_copy does not share the inner lists.
        graph_copy = defaultdict(list)
        for kmer, outs in self.graph.items():
            graph_copy[kmer] = outs.copy()

        #helper function to choose start node:
        def nodes_with_edges():
            for node, outs in graph_copy.items():
                if len(outs) > 0:
                    return node

        def pick_start():
            candidates = notes_with_edges()
            if not candidates:
                return None

            starts = []
            for node in candidates:
                out_degree = len(graph_copy[node])
                in_degree = self.suffix_counts.get(node, 0)
                #Exactly one node with out-degree = in-degree + 1 (start node).
                if out_degree - in_degree == 1:
                    starts.append(node)

            if starts:
                if seed is not None:
                    random.seed(seed)
                    return random.choice(starts)
                #else return the first one
                return starts[0]

        contigs = []

        while True:
            start = pick_start()
            if start is None:
                break

            tour_rev = self.eulerian_walk(start, graph_copy, seed=seed)
            tour = list(reversed(tour_rev))

            if len(tour) >=2:
                contigs.append(self.tour_to_sequence(tour))

        return contigs


    def tour_to_sequence(self, tour):
        """Convert a tour of (k-1)-mers into a DNA sequence.

        Args:
            tour (list): List of (k-1)-mer strings in order.

        Returns:
            str: Assembled DNA sequence.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> tour = ['ATG', 'TGG', 'GGC', 'GCG']
            >>> dbg.tour_to_sequence(tour)
            'ATGGCG'
        """
        if not tour:
            return ""

        seq = tour[0]
        for node in tour[1:]:
            seq += node[-1]
        return seq

    def get_assembly_stats(self, contigs):
        """Calculate assembly statistics for assembled contigs.

        Args:
            contigs (list): List of contig sequences.

        Returns:
            dict: Dictionary containing assembly statistics:
                - num_contigs: Total number of contigs
                - total_length: Total assembled sequence length
                - longest_contig: Length of longest contig
                - shortest_contig: Length of shortest contig
                - mean_length: Mean contig length
                - n50: N50 statistic

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA", "CCCCCCCCCC"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> stats = dbg.get_assembly_stats(contigs)
            >>> stats['num_contigs']
            3
        """
        stats = {
            "num_contigs": 0,
            "total_length": 0,
            "longest_contig": 0,
            "shortest_contig": 0,
            "mean_length": 0,
            "n50": 0,
        }

        if not contigs:
            return stats

        lengths = [len(each) for each in contigs]


        stats["num_contigs"] = len(contigs)
        stats["total_length"] = sum(lengths)
        stats["longest_contig"] = max(lengths)
        stats["shortest_contig"] = min(lengths)
        stats["mean_length"] = sum(lengths) / len(lengths)

        half = sum(lengths) / 2
        cumulative = 0
        for contig_length in sorted(lengths, reverse=True):
            cumulative += contig_length
            if cumulative >= half:
                stats['n50'] = contig_length
                break

        return stats

    def write_fasta(self, contigs, filename):
        """Write assembled contigs to a FASTA file.

        Args:
            contigs (list): List of contig sequences.
            filename (str): Output FASTA filename.

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.write_fasta(contigs, "output.fasta")
        """
        with open(filename, "w", encoding='uft-8') as outfile:
            for i, contig in enumerate(contigs):
                outfile.write(f">Contig_{i}\n")
                outfile.write(contig + "\n")


---
# Toy Example

In [4]:
print("="*60)
print("EXAMPLE 1: Assembling 9 overlapping reads into one contig")
print("="*60)

toy_reads_1 = [
    "ATGGCGTACG",  # Read 1
    "GGCGTACGTT",  # Read 2: overlaps with Read 1
    "CGTACGTTAC",  # Read 3: overlaps with Read 2
    "TACGTTACCA",  # Read 4: overlaps with Read 3
    "CGTTACCATG",  # Read 5: overlaps with Read 4
    "TTACCATGGG",  # Read 6: overlaps with Read 5
    "ACCATGGGCC",  # Read 7: overlaps with Read 6
    "CATGGGCCTA",  # Read 8: overlaps with Read 7
    "TGGGCCTAAA"   # Read 9: overlaps with Read 8
]

print(f"\nInput: {len(toy_reads_1)} reads")
print("First read:  ", toy_reads_1[0])
print("Last read:   ", toy_reads_1[-1])
print(f"\nBuilding De Bruijn graph with k=6...")

EXAMPLE 1: Assembling 9 overlapping reads into one contig

Input: 9 reads
First read:   ATGGCGTACG
Last read:    TGGGCCTAAA

Building De Bruijn graph with k=6...


---
# Real data
This section utilizes a FASTQ file with 10 million "perfect" 150 bp reads simulated
from the mouse genome (`GRCm39`) and tasks your program to ingest, assemble, and traverse
the genome with statistical output report.

In [ ]:
"""Driver program for mouse genome assembly using De Bruijn graphs.

This script demonstrates how to use the completed DeBruijnGraph class to
assemble 10 million perfect 150bp single-end reads from the mouse genome.

Usage:
    Run all cells in order in a Jupyter notebook.

Expected input:
    - File: mouse_SE_150bp.fq
    - Format: FASTQ
    - Reads: 10 million perfect 150bp single-end reads
    - Source: Simulated from mouse genome using wgsim

Output:
    - mouse_assembly.fasta: Assembled contigs
    - mouse_assembly_stats.txt: Detailed statistics
"""

import time
from datetime import datetime


def write_statistics_file(
    stats_file,
    input_file,
    num_reads,
    avg_read_length,
    k_mer_size,
    random_seed,
    num_nodes,
    num_edges,
    stats,
    contig_lengths,
    timing,
    coverage_estimate,
    assembly_fraction
):
    """Write comprehensive assembly statistics to text file.
    
    Args:
        stats_file (str): Output filename.
        input_file (str): Input FASTQ filename.
        num_reads (int): Number of reads processed.
        avg_read_length (float): Average read length.
        k_mer_size (int): K-mer size used.
        random_seed (int): Random seed used.
        num_nodes (int): Number of graph nodes.
        num_edges (int): Number of graph edges.
        stats (dict): Assembly statistics.
        contig_lengths (list): Sorted list of contig lengths.
        timing (dict): Timing information.
        coverage_estimate (float): Estimated sequencing coverage.
        assembly_fraction (float): Assembly size as % of genome.
    """
    with open(stats_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MOUSE GENOME ASSEMBLY STATISTICS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input File: {input_file}\n")
        f.write(f"K-mer Size: {k_mer_size}\n")
        f.write(f"Random Seed: {random_seed}\n\n")
        
        f.write("-"*80 + "\n")
        f.write("INPUT DATA\n")
        f.write("-"*80 + "\n")
        f.write(f"Number of reads:         {num_reads:,}\n")
        f.write(f"Average read length:     {avg_read_length:.1f} bp\n")
        f.write(f"Total sequencing data:   {num_reads * avg_read_length:,.0f} bp\n")
        f.write(f"Estimated coverage:      {coverage_estimate:.1f}x\n")
        f.write(f"Read time:               {timing['read_time']:.2f} seconds\n\n")
        
        f.write("-"*80 + "\n")
        f.write("DE BRUIJN GRAPH CONSTRUCTION\n")
        f.write("-"*80 + "\n")
        f.write(f"Graph nodes:             {num_nodes:,}\n")
        f.write(f"Graph edges:             {num_edges:,}\n")
        f.write(f"Average out-degree:      {num_edges/num_nodes:.2f}\n")
        f.write(f"Construction time:       {timing['graph_time']:.2f} seconds\n\n")
        
        f.write("-"*80 + "\n")
        f.write("ASSEMBLY RESULTS\n")
        f.write("-"*80 + "\n")
        f.write(f"Number of contigs:       {stats['num_contigs']:,}\n")
        f.write(f"Total assembly length:   {stats['total_length']:,} bp\n")
        f.write(f"Assembly vs. genome:     {assembly_fraction:.2f}%\n")
        f.write(f"Longest contig:          {stats['longest_contig']:,} bp\n")
        f.write(f"Shortest contig:         {stats['shortest_contig']:,} bp\n")
        f.write(f"Mean contig length:      {stats['mean_length']:,.1f} bp\n")
        f.write(f"N50:                     {stats['n50']:,} bp\n")
        f.write(f"Assembly time:           {timing['assembly_time']:.2f} seconds\n\n")
        
        f.write("-"*80 + "\n")
        f.write("TOP 20 LONGEST CONTIGS\n")
        f.write("-"*80 + "\n")
        for i, length in enumerate(contig_lengths[:20], 1):
            f.write(f"{i:3d}. {length:10,} bp\n")
        f.write("\n")
        
        f.write("-"*80 + "\n")
        f.write("CONTIG LENGTH DISTRIBUTION\n")
        f.write("-"*80 + "\n")
        bins = [
            (">100kb", sum(1 for x in contig_lengths if x > 100000)),
            (">50kb", sum(1 for x in contig_lengths if x > 50000)),
            (">10kb", sum(1 for x in contig_lengths if x > 10000)),
            (">5kb", sum(1 for x in contig_lengths if x > 5000)),
            (">1kb", sum(1 for x in contig_lengths if x > 1000)),
            (">500bp", sum(1 for x in contig_lengths if x > 500)),
        ]
        for bin_name, count in bins:
            f.write(f"Contigs {bin_name:8s}:     {count:,}\n")
        f.write("\n")
        
        f.write("-"*80 + "\n")
        f.write("TIMING SUMMARY\n")
        f.write("-"*80 + "\n")
        total_time = timing['total_time']
        f.write(f"Read time:               {timing['read_time']:8.2f} seconds "
                f"({timing['read_time']/total_time*100:5.1f}%)\n")
        f.write(f"Graph construction:      {timing['graph_time']:8.2f} seconds "
                f"({timing['graph_time']/total_time*100:5.1f}%)\n")
        f.write(f"Assembly:                {timing['assembly_time']:8.2f} seconds "
                f"({timing['assembly_time']/total_time*100:5.1f}%)\n")
        f.write(f"Total time:              {total_time:8.2f} seconds "
                f"({total_time/60:.2f} minutes)\n\n")
        
        f.write("="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")


def assemble_mouse_genome(
    input_file="data/mouse_SE_150bp.fq",
    output_fasta="mouse_assembly.fasta", 
    stats_file="mouse_assembly_stats.txt",
    k_mer_size = 51,
    random_seed = None
):
    """Main driver function for mouse genome assembly.
    
    This function orchestrates the complete assembly pipeline:
    1. Load FASTQ reads
    2. Build De Bruijn graph
    3. Assemble contigs
    4. Calculate statistics
    5. Write output files
    
    Returns:
        dict: Dictionary containing assembly results including:
            - dbg: DeBruijnGraph object
            - contigs: List of assembled sequences
            - stats: Assembly statistics dictionary
            - timing: Performance timing information
    """
    print("="*80)
    print("MOUSE GENOME ASSEMBLY PIPELINE")
    print("="*80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    timing = {}
    
    print("STEP 1: Loading sequencing reads")
    print("-"*80)
    print(f"Input file: {input_file}")
    print(f"Expected: 10 million reads, 150bp each")
    print()
    
    start_time = time.time()
    reads = read_fastq(input_file)
    timing['read_time'] = time.time() - start_time
    
    num_reads = len(reads)
    total_bases = sum(len(read) for read in reads)
    avg_read_length = total_bases / num_reads if num_reads > 0 else 0
    
    print(f"Reads loaded: {num_reads:,}")
    print(f"Total bases: {total_bases:,} bp")
    print(f"Average read length: {avg_read_length:.1f} bp")
    print(f"Time elapsed: {timing['read_time']:.2f} seconds")
    
    if num_reads > 0:
        print(f"\nSample reads:")
        print(f"  First: {reads[0][:80]}...")
        print(f"  Last:  {reads[-1][:80]}...")
    print()
    
    print("STEP 2: Building De Bruijn graph")
    print("-"*80)
    print(f"K-mer size: {k_mer_size}")
    print(f"Building graph from {num_reads:,} reads...")
    print()
    
    start_time = time.time()
    dbg = DeBruijnGraph(reads, k=k_mer_size)
    timing['graph_time'] = time.time() - start_time
    
    # Calculate graph statistics
    num_nodes = len(dbg.graph)
    num_edges = sum(len(neighbors) for neighbors in dbg.graph.values())
    avg_degree = num_edges / num_nodes if num_nodes > 0 else 0
    
    print(f"Graph construction complete!")
    print(f"  Nodes (unique {k_mer_size-1}-mers): {num_nodes:,}")
    print(f"  Edges (k-mer transitions): {num_edges:,}")
    print(f"  Average out-degree: {avg_degree:.2f}")
    print(f"Time elapsed: {timing['graph_time']:.2f} seconds")
    print()
    
    print("STEP 3: Assembling contigs")
    print("-"*80)
    print(f"Finding connected components and traversing graph...")
    print(f"Random seed: {random_seed} (for reproducibility)")
    print()
    
    start_time = time.time()
    contigs = dbg.assemble_contigs(seed=random_seed)
    timing['assembly_time'] = time.time() - start_time
    
    print(f"Assembly complete!")
    print(f"  Contigs generated: {len(contigs):,}")
    print(f"Time elapsed: {timing['assembly_time']:.2f} seconds")
    print()
    
    print("STEP 4: Calculating assembly statistics")
    print("-"*80)
    
    stats = dbg.get_assembly_stats(contigs)
    
    print(f"Assembly Statistics:")
    print(f"  Number of contigs:     {stats['num_contigs']:,}")
    print(f"  Total assembly length: {stats['total_length']:,} bp")
    print(f"  Longest contig:        {stats['longest_contig']:,} bp")
    print(f"  Shortest contig:       {stats['shortest_contig']:,} bp")
    print(f"  Mean contig length:    {stats['mean_length']:,.1f} bp")
    print(f"  N50:                   {stats['n50']:,} bp")
    print()
    
    # Display distribution of contig lengths
    contig_lengths = sorted([len(c) for c in contigs], reverse=True)
    
    print(f"Contig Length Distribution:")
    print(f"  Top 10 longest contigs:")
    for i, length in enumerate(contig_lengths[:10], 1):
        print(f"    {i:2d}. {length:,} bp")
    print()
    
    # Calculate coverage estimate
    genome_size_estimate = 2700000000  # Mouse genome ~2.7 Gbp
    coverage_estimate = (num_reads * avg_read_length) / genome_size_estimate
    assembly_fraction = (stats['total_length'] / genome_size_estimate) * 100
    
    print(f"Genome Coverage Analysis:")
    print(f"  Mouse genome size (expected): ~{genome_size_estimate:,} bp")
    print(f"  Estimated sequencing coverage: {coverage_estimate:.1f}x")
    print(f"  Assembly size vs. genome: {assembly_fraction:.1f}%")
    print()
    
    print("STEP 5: Writing output files")
    print("-"*80)
    
    # Write assembled contigs to FASTA
    dbg.write_fasta(contigs, output_fasta)
    print(f"✓ Contigs written to: {output_fasta}")
    
    # Write detailed statistics
    write_statistics_file(
        stats_file,
        input_file,
        num_reads,
        avg_read_length,
        k_mer_size,
        random_seed,
        num_nodes,
        num_edges,
        stats,
        contig_lengths,
        timing,
        coverage_estimate,
        assembly_fraction
    )
    print(f"✓ Statistics written to: {stats_file}")
    print()
    
    timing['total_time'] = (timing['read_time'] + timing['graph_time'] + 
                           timing['assembly_time'])
    
    print("="*80)
    print("ASSEMBLY COMPLETE")
    print("="*80)
    print(f"Total time: {timing['total_time']:.2f} seconds "
          f"({timing['total_time']/60:.2f} minutes)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    print(f"Summary:")
    print(f"  • Processed {num_reads:,} reads ({total_bases:,} bp)")
    print(f"  • Built graph with {num_nodes:,} nodes and {num_edges:,} edges")
    print(f"  • Assembled {stats['num_contigs']:,} contigs")
    print(f"  • Total assembly: {stats['total_length']:,} bp (N50: {stats['n50']:,} bp)")
    print(f"  • Output files: {output_fasta}, {stats_file}")
    print()
    
    return {
        'dbg': dbg,
        'contigs': contigs,
        'stats': stats,
        'timing': timing,
        'graph_stats': {
            'num_nodes': num_nodes,
            'num_edges': num_edges,
            'avg_degree': avg_degree
        },
        'coverage': coverage_estimate
    }


print("\n" + "="*80)
print("MOUSE GENOME DE BRUIJN GRAPH ASSEMBLY")
print("Student Assignment Driver Program")
print("="*80 + "\n")

# Run the assembly pipeline
result = assemble_mouse_genome()

# Display sample contigs
print("="*80)
print("SAMPLE ASSEMBLED CONTIGS")
print("="*80 + "\n")

contigs = result['contigs']
for i in range(min(5, len(contigs))):
    contig = contigs[i]
    preview = contig[:100] + "..." if len(contig) > 100 else contig
    print(f">contig_{i+1} length={len(contig)}")
    print(preview)
    print()

print("="*80)
print("✓ ASSEMBLY COMPLETE")
print("="*80)
print(f"\nResults saved to:")
print(f"  • mouse_assembly.fasta - {result['stats']['num_contigs']:,} assembled contigs")
print(f"  • mouse_assembly_stats.txt - Detailed assembly statistics")
print(f"\nKey metrics:")
print(f"  • Total assembly: {result['stats']['total_length']:,} bp")
print(f"  • N50: {result['stats']['n50']:,} bp")
print(f"  • Longest contig: {result['stats']['longest_contig']:,} bp")
print(f"  • Coverage: {result['coverage']:.1f}x")
print()


MOUSE GENOME DE BRUIJN GRAPH ASSEMBLY
Student Assignment Driver Program

MOUSE GENOME ASSEMBLY PIPELINE
Start time: 2026-02-24 02:24:41

STEP 1: Loading sequencing reads
--------------------------------------------------------------------------------
Input file: data/mouse_SE_150bp.fq
Expected: 10 million reads, 150bp each

Reads loaded: 10,000,001
Total bases: 1,500,000,150 bp
Average read length: 150.0 bp
Time elapsed: 3.80 seconds

Sample reads:
  First: AATCAGGGAGAGACTGGGAGAAGTGGAGGGAGTGGAAATCACAGGAGGGATGTAATATATGAGAGAAGAATAAATGTAAG...
  Last:  TTTAGGCATTCCGGTGTTGGGTTAACAGAGAAGTTATAGGTGGATTATTTATAGTGTGATTATTGCCTATAGTCTGATTA...

STEP 2: Building De Bruijn graph
--------------------------------------------------------------------------------
K-mer size: 51
Building graph from 10,000,001 reads...

